In [0]:
from pyspark.sql import functions as F

silver_safety_incidents = spark.table(
    "workspace.transportation_analytics.silver_safety_incidents"
)

silver_drivers = spark.table(
    "workspace.transportation_analytics.silver_drivers"
)

silver_trucks = spark.table(
    "workspace.transportation_analytics.silver_trucks"
)

silver_routes = spark.table(
    "workspace.transportation_analytics.silver_routes"
)

print("Safety Metrics Silver tables loaded successfully")

In [0]:
silver_safety_incidents.printSchema()
silver_trucks.printSchema()
silver_routes.printSchema()
silver_drivers.printSchema()

In [0]:
gold_safety_metrics = (
    silver_safety_incidents
    .groupBy("driver_id")
    .agg(
        F.countDistinct("incident_id").alias("total_incidents"),
        F.sum(
            F.when(F.col("preventable_flag") == True, 1).otherwise(0)
        ).alias("preventable_accidents"),
        F.sum(
            F.when(F.col("at_fault_flag") == True, 1).otherwise(0)
        ).alias("at_fault_incidents"),
        F.sum(
            F.when(F.col("injury_flag") == True, 1).otherwise(0)
        ).alias("injury_incidents"),
        F.sum("vehicle_damage_cost").alias("total_vehicle_damage_cost"),
        F.sum("cargo_damage_cost").alias("total_cargo_damage_cost"),
        F.sum("claim_amount").alias("total_claim_amount")
    )
)

display(gold_safety_metrics)

In [0]:
gold_safety_metrics = (
    gold_safety_metrics
    .join(
        silver_drivers.select(
            "driver_id",
            "first_name",
            "last_name",
            "employment_status"
        ),
        on="driver_id",
        how="left"
    )
)

display(gold_safety_metrics)

In [0]:
silver_trips = spark.table(
    "workspace.transportation_analytics.silver_trips"
)

silver_loads = spark.table(
    "workspace.transportation_analytics.silver_loads"
)

print("Trips and Loads tables loaded successfully")

In [0]:
incident_route = (
    silver_safety_incidents
    .join(
        silver_trips.select(
            "trip_id",
            "load_id"
        ),
        on="trip_id",
        how="left"
    )
    .join(
        silver_loads.select(
            "load_id",
            "route_id"
        ),
        on="load_id",
        how="left"
    )
)

display(
    incident_route.select(
        "incident_id",
        "driver_id",
        "truck_id",
        "incident_date",
        "incident_type",
        "preventable_flag",
        "at_fault_flag",
        "route_id"
    )
)

In [0]:
gold_safety_metrics = (
    incident_route
    .groupBy(
        "driver_id",
        "truck_id",
        "route_id"
    )
    .agg(
        F.countDistinct("incident_id").alias("total_incidents"),
        F.sum(
            F.when(F.col("preventable_flag") == True, 1).otherwise(0)
        ).alias("preventable_accidents"),
        F.sum(
            F.when(F.col("at_fault_flag") == True, 1).otherwise(0)
        ).alias("at_fault_incidents"),
        F.sum(
            F.when(F.col("injury_flag") == True, 1).otherwise(0)
        ).alias("injury_incidents"),
        F.sum("vehicle_damage_cost").alias("total_vehicle_damage_cost"),
        F.sum("cargo_damage_cost").alias("total_cargo_damage_cost"),
        F.sum("claim_amount").alias("total_claim_amount")
    )
)

display(gold_safety_metrics)

In [0]:
gold_safety_metrics = (
    gold_safety_metrics
    .join(
        silver_drivers.select(
            "driver_id",
            "first_name",
            "last_name",
            "employment_status"
        ),
        on="driver_id",
        how="left"
    )
    .join(
        silver_trucks.select(
            "truck_id",
            "unit_number",
            "make",
            "model_year",
            "status"
        ),
        on="truck_id",
        how="left"
    )
)

display(gold_safety_metrics)

In [0]:
display(
    gold_safety_metrics.select(
        "driver_id",
        "first_name",
        "last_name",
        "truck_id",
        "unit_number",
        "route_id",
        "total_incidents",
        "preventable_accidents",
        "at_fault_incidents",
        "injury_incidents",
        "total_vehicle_damage_cost",
        "total_cargo_damage_cost",
        "total_claim_amount"
    )
)

In [0]:
gold_safety_metrics.limit(0).write \
    .format("delta") \
    .saveAsTable(
        "workspace.transportation_analytics.gold_safety_metrics"
    )

print("Empty Gold Safety Metrics table created successfully")

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_safety_metrics"
)

target.alias("t").merge(
    gold_safety_metrics.alias("s"),
    """
    t.driver_id = s.driver_id
    AND t.truck_id = s.truck_id
    AND t.route_id = s.route_id
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Safety Metrics table updated using MERGE")

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_safety_metrics"
    )
)

In [0]:
display(
    gold_safety_metrics
    .groupBy(
        "driver_id",
        "first_name",
        "last_name"
    )
    .agg(
        F.sum("preventable_accidents").alias(
            "total_preventable_accidents"
        )
    )
    .orderBy(F.desc("total_preventable_accidents"))
)

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_safety_metrics
    .groupBy("truck_id")
    .agg(
        F.sum("total_incidents").alias("total_incidents")
    )
    .orderBy(F.desc("total_incidents"))
)

Databricks visualization. Run in Databricks to view.

In [0]:
gold_safety_metrics = (
    incident_route
    .withColumn(
        "incident_month",
        F.date_trunc("month", F.col("incident_date"))
    )
    .groupBy(
        "driver_id",
        "truck_id",
        "route_id",
        "incident_month"
    )
    .agg(
        F.countDistinct("incident_id").alias("total_incidents"),
        F.sum(
            F.when(F.col("preventable_flag") == True, 1).otherwise(0)
        ).alias("preventable_accidents"),
        F.sum(
            F.when(F.col("at_fault_flag") == True, 1).otherwise(0)
        ).alias("at_fault_incidents"),
        F.sum(
            F.when(F.col("injury_flag") == True, 1).otherwise(0)
        ).alias("injury_incidents"),
        F.sum("vehicle_damage_cost").alias("total_vehicle_damage_cost"),
        F.sum("cargo_damage_cost").alias("total_cargo_damage_cost"),
        F.sum("claim_amount").alias("total_claim_amount")
    )
)

display(gold_safety_metrics)

In [0]:
spark.sql("""
ALTER TABLE workspace.transportation_analytics.gold_safety_metrics
ADD COLUMNS (
    incident_month TIMESTAMP
)
""")

print("incident_month column added successfully")

In [0]:
gold_safety_metrics = (
    gold_safety_metrics
    .join(
        silver_drivers.select(
            "driver_id",
            "first_name",
            "last_name",
            "employment_status"
        ),
        on="driver_id",
        how="left"
    )
    .join(
        silver_trucks.select(
            "truck_id",
            "unit_number",
            "make",
            "model_year",
            "status"
        ),
        on="truck_id",
        how="left"
    )
)

display(gold_safety_metrics)

In [0]:
target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_safety_metrics"
)

target.alias("t").merge(
    gold_safety_metrics.alias("s"),
    """
    t.driver_id = s.driver_id
    AND t.truck_id = s.truck_id
    AND t.route_id = s.route_id
    AND t.incident_month = s.incident_month
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Safety Metrics table updated successfully with time period")

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_safety_metrics"
    )
)

In [0]:
trip_counts = (
    silver_trips
    .withColumn(
        "trip_month",
        F.date_trunc("month", F.col("dispatch_date"))
    )
    .groupBy(
        "driver_id",
        "truck_id",
        "load_id",
        "trip_month"
    )
    .agg(
        F.countDistinct("trip_id").alias("total_trips")
    )
)

display(trip_counts)

In [0]:
gold_safety_metrics = (
    gold_safety_metrics
    .join(
        trip_counts,
        (
            (gold_safety_metrics.driver_id == trip_counts.driver_id) &
            (gold_safety_metrics.truck_id == trip_counts.truck_id) &
            (gold_safety_metrics.incident_month == trip_counts.trip_month)
        ),
        how="left"
    )
    .drop(trip_counts.driver_id)
    .drop(trip_counts.truck_id)
    .drop("trip_month")
)

gold_safety_metrics = (
    gold_safety_metrics
    .withColumn(
        "incident_rate_per_1000_trips",
        F.round(
            F.col("total_incidents") /
            F.col("total_trips") * 1000,
            2
        )
    )
)

display(gold_safety_metrics)

In [0]:
trip_counts = (
    silver_trips
    .withColumn(
        "trip_month",
        F.date_trunc("month", "dispatch_date")
    )
    .groupBy(
        "driver_id",
        "truck_id",
        "trip_month"
    )
    .agg(
        F.countDistinct("trip_id").alias("total_trips")
    )
)

display(trip_counts)

In [0]:
gold_safety_metrics = gold_safety_metrics.join(
    trip_counts,
    ["driver_id", "truck_id"],
    "left"
)

In [0]:
gold_safety_metrics = gold_safety_metrics.drop(trip_counts.total_trips)

In [0]:
gold_safety_metrics = spark.table(
    "workspace.transportation_analytics.gold_safety_metrics"
)

display(gold_safety_metrics)

In [0]:
display(
    gold_safety_metrics.select(
        "driver_id",
        "truck_id",
        "route_id",
        "incident_month",
        "total_incidents",
        "total_trips",
        "incident_rate_per_1000_trips"
    )
)

In [0]:
display(
    gold_safety_metrics
    .groupBy("incident_month")
    .agg(
        F.sum("total_incidents").alias("total_incidents"),
        F.sum("preventable_accidents").alias("preventable_accidents")
    )
    .orderBy("incident_month")
)

In [0]:
display(
    gold_safety_metrics
    .groupBy("driver_id")
    .agg(
        F.sum("total_incidents").alias("total_incidents"),
        F.sum("total_trips").alias("total_trips"),
        F.round(
            F.sum("total_incidents") /
            F.sum("total_trips") * 1000,
            2
        ).alias("incident_rate_per_1000_trips")
    )
    .orderBy(F.desc("incident_rate_per_1000_trips"))
)

In [0]:
display(
    gold_safety_metrics
    .groupBy("truck_id")
    .agg(
        F.sum("total_incidents").alias("total_incidents"),
        F.sum("total_trips").alias("total_trips"),
        F.round(
            F.sum("total_incidents") /
            F.sum("total_trips") * 1000,
            2
        ).alias("incident_rate_per_1000_trips")
    )
    .orderBy(F.desc("incident_rate_per_1000_trips"))
)

In [0]:
display(
    gold_safety_metrics
    .groupBy("incident_month")
    .agg(
        F.sum("total_incidents").alias("total_incidents"),
        F.sum("preventable_accidents").alias("preventable_accidents")
    )
    .orderBy("incident_month")
)

Databricks visualization. Run in Databricks to view.